# IBM Telco Customer Churn: Exploratory Data Analysis

This notebook performs a complete exploratory data analysis of the IBM Telco Customer Churn dataset using pandas and Plotly. The goal is to understand customer churn patterns across contract type, tenure, monthly charges, service usage, and numeric relationships before model development.

The notebook is designed to run from either the project root or the `notebooks/` directory.

## 1. Setup

Import the analysis stack, configure display options, and locate the project data file.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

px.defaults.template = "plotly_white"
px.defaults.color_discrete_sequence = px.colors.qualitative.Set2

In [ ]:
# Locate project root robustly whether the notebook is run from / or /notebooks.
candidate_roots = [Path.cwd(), Path.cwd().parent]
data_filename = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

PROJECT_ROOT = next(
    (root for root in candidate_roots if (root / "data" / data_filename).exists()),
    Path.cwd(),
)
DATA_PATH = PROJECT_ROOT / "data" / data_filename

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Place {data_filename} in the project's data/ directory."
    )

DATA_PATH

## 2. Load Data

Load the raw CSV, keep a copy unchanged, and create an analysis dataframe with type-safe fields.

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()

# TotalCharges is stored as text in the original dataset because a small number of rows contain blanks.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["SeniorCitizenLabel"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})
df["ChurnFlag"] = df["Churn"].map({"No": 0, "Yes": 1})

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
df.head()

## 3. Dataset Overview

Review the schema, completeness, uniqueness, and summary statistics.

In [ ]:
overview = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(dtype) for dtype in df.dtypes],
    "non_null": df.notna().sum().values,
    "missing": df.isna().sum().values,
    "missing_pct": (df.isna().mean().values * 100).round(2),
    "unique_values": df.nunique(dropna=True).values,
})

overview

In [ ]:
df.describe(include="all").T

In [ ]:
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
categorical_cols = [
    col for col in df.columns
    if col not in numeric_cols + ["customerID", "ChurnFlag"]
]

display(Markdown(
    f"""
**Observations**

- The dataset contains **{df.shape[0]:,} customers** and **{df_raw.shape[1]:,} original columns**.
- Key numeric fields for churn behavior are **tenure**, **MonthlyCharges**, and **TotalCharges**.
- Most predictors are categorical service, billing, or account attributes, which will need encoding before model training.
- `customerID` is an identifier and should not be used as a predictive feature.
"""
))

## 4. Missing Value Analysis

Check raw missing values, blank strings, and the post-conversion missing values in `TotalCharges`.

In [ ]:
raw_blank_counts = df_raw.apply(
    lambda series: series.astype(str).str.strip().eq("").sum()
).rename("blank_strings")

missing_analysis = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "blank_strings_in_raw": raw_blank_counts,
}).sort_values(["missing_count", "blank_strings_in_raw"], ascending=False)

missing_analysis

In [ ]:
missing_to_plot = missing_analysis.reset_index().rename(columns={"index": "column"})
missing_to_plot = missing_to_plot[
    (missing_to_plot["missing_count"] > 0) | (missing_to_plot["blank_strings_in_raw"] > 0)
]

if missing_to_plot.empty:
    fig = go.Figure()
    fig.add_annotation(text="No missing values or blank strings found", x=0.5, y=0.5, showarrow=False)
    fig.update_layout(title="Missing Value Analysis", height=350)
else:
    fig = px.bar(
        missing_to_plot,
        x="column",
        y=["missing_count", "blank_strings_in_raw"],
        barmode="group",
        title="Missing Values and Raw Blank Strings by Column",
        labels={"value": "count", "variable": "issue type"},
    )

fig.show()

In [ ]:
display(Markdown(
    f"""
**Observations**

- After type conversion, **{int(df.isna().sum().sum()):,} missing values** are present across the analysis dataframe.
- The affected converted field is primarily `TotalCharges`, where blank raw strings become numeric missing values.
- Rows with missing `TotalCharges` represent customers with no accumulated charges yet, so they should be handled deliberately before training.
- No broad missingness pattern appears across the service and account fields.
"""
))

## 5. Duplicate Checking

Check both full-row duplicates and duplicate customer identifiers.

In [ ]:
full_duplicate_count = int(df.duplicated().sum())
customer_duplicate_count = int(df["customerID"].duplicated().sum())

duplicate_summary = pd.DataFrame({
    "check": ["Full row duplicates", "Duplicate customerID values"],
    "count": [full_duplicate_count, customer_duplicate_count],
})

duplicate_summary

In [ ]:
display(Markdown(
    f"""
**Observations**

- Full duplicate rows found: **{full_duplicate_count:,}**.
- Duplicate `customerID` values found: **{customer_duplicate_count:,}**.
- This indicates each row can be treated as one unique customer record for EDA and model preparation.
"""
))

## 6. Churn Distribution

Measure the overall churn class balance.

In [ ]:
churn_counts = (
    df["Churn"]
    .value_counts()
    .rename_axis("Churn")
    .reset_index(name="Customers")
)
churn_counts["Share"] = churn_counts["Customers"] / churn_counts["Customers"].sum()
churn_counts

In [ ]:
fig = px.pie(
    churn_counts,
    names="Churn",
    values="Customers",
    hole=0.45,
    title="Customer Churn Distribution",
)
fig.update_traces(textposition="inside", textinfo="percent+label")
fig.show()

In [ ]:
fig = px.bar(
    churn_counts,
    x="Churn",
    y="Customers",
    color="Churn",
    text="Customers",
    title="Churn Class Counts",
)
fig.update_traces(texttemplate="%{text:,}", textposition="outside")
fig.update_layout(showlegend=False, yaxis_title="Customers")
fig.show()

In [ ]:
churn_rate = df["ChurnFlag"].mean()
majority_class = churn_counts.sort_values("Customers", ascending=False).iloc[0]

display(Markdown(
    f"""
**Observations**

- Overall churn rate is **{churn_rate:.1%}**.
- The majority class is **{majority_class['Churn']}**, with **{int(majority_class['Customers']):,} customers**.
- The classes are imbalanced but still large enough to compare churn patterns across customer segments.
"""
))

## 7. Contract Type Analysis

Compare churn behavior by contract duration.

In [ ]:
contract_analysis = (
    df.groupby(["Contract", "Churn"], observed=True)
    .size()
    .reset_index(name="Customers")
)

contract_rate = (
    df.groupby("Contract", observed=True)
    .agg(Customers=("customerID", "count"), ChurnRate=("ChurnFlag", "mean"))
    .reset_index()
    .sort_values("ChurnRate", ascending=False)
)

contract_rate["ChurnRatePct"] = (contract_rate["ChurnRate"] * 100).round(2)
contract_rate

In [ ]:
fig = px.bar(
    contract_analysis,
    x="Contract",
    y="Customers",
    color="Churn",
    barmode="group",
    title="Churn Counts by Contract Type",
)
fig.update_layout(xaxis_title="Contract Type", yaxis_title="Customers")
fig.show()

In [ ]:
fig = px.bar(
    contract_rate,
    x="Contract",
    y="ChurnRatePct",
    color="Contract",
    text="ChurnRatePct",
    title="Churn Rate by Contract Type",
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(showlegend=False, yaxis_title="Churn Rate (%)", xaxis_title="Contract Type")
fig.show()

In [ ]:
highest_contract = contract_rate.iloc[0]
lowest_contract = contract_rate.iloc[-1]

display(Markdown(
    f"""
**Observations**

- **{highest_contract['Contract']}** customers have the highest churn rate at **{highest_contract['ChurnRate']:.1%}**.
- **{lowest_contract['Contract']}** customers have the lowest churn rate at **{lowest_contract['ChurnRate']:.1%}**.
- Longer contractual commitment appears strongly associated with lower churn risk.
"""
))

## 8. Tenure Analysis

Analyze how customer lifetime relates to churn.

In [ ]:
tenure_summary = (
    df.groupby("Churn", observed=True)["tenure"]
    .describe()
    .round(2)
)
tenure_summary

In [ ]:
fig = px.histogram(
    df,
    x="tenure",
    color="Churn",
    nbins=36,
    marginal="box",
    barmode="overlay",
    opacity=0.75,
    title="Tenure Distribution by Churn Status",
)
fig.update_layout(xaxis_title="Tenure (months)", yaxis_title="Customers")
fig.show()

In [ ]:
tenure_bins = [-0.1, 12, 24, 36, 48, 60, 72]
tenure_labels = ["0-12", "13-24", "25-36", "37-48", "49-60", "61-72"]

df["TenureGroup"] = pd.cut(df["tenure"], bins=tenure_bins, labels=tenure_labels)

tenure_group_analysis = (
    df.groupby("TenureGroup", observed=True)
    .agg(Customers=("customerID", "count"), ChurnRate=("ChurnFlag", "mean"))
    .reset_index()
)
tenure_group_analysis["ChurnRatePct"] = (tenure_group_analysis["ChurnRate"] * 100).round(2)
tenure_group_analysis

In [ ]:
fig = px.line(
    tenure_group_analysis,
    x="TenureGroup",
    y="ChurnRatePct",
    markers=True,
    title="Churn Rate by Tenure Group",
)
fig.update_traces(line_width=3)
fig.update_layout(xaxis_title="Tenure Group (months)", yaxis_title="Churn Rate (%)")
fig.show()

In [ ]:
median_tenure_churned = df.loc[df["Churn"] == "Yes", "tenure"].median()
median_tenure_retained = df.loc[df["Churn"] == "No", "tenure"].median()
first_tenure_group = tenure_group_analysis.iloc[0]
last_tenure_group = tenure_group_analysis.iloc[-1]

display(Markdown(
    f"""
**Observations**

- Median tenure for churned customers is **{median_tenure_churned:.0f} months**, compared with **{median_tenure_retained:.0f} months** for retained customers.
- The earliest tenure group, **{first_tenure_group['TenureGroup']} months**, has a churn rate of **{first_tenure_group['ChurnRate']:.1%}**.
- The latest tenure group, **{last_tenure_group['TenureGroup']} months**, has a churn rate of **{last_tenure_group['ChurnRate']:.1%}**.
- Churn risk is concentrated among newer customers, making onboarding and early-life retention important business levers.
"""
))

## 9. Monthly Charges Analysis

Evaluate whether customers with higher monthly charges churn more often.

In [ ]:
monthly_summary = (
    df.groupby("Churn", observed=True)["MonthlyCharges"]
    .describe()
    .round(2)
)
monthly_summary

In [ ]:
fig = px.histogram(
    df,
    x="MonthlyCharges",
    color="Churn",
    nbins=40,
    marginal="box",
    barmode="overlay",
    opacity=0.7,
    title="Monthly Charges Distribution by Churn Status",
)
fig.update_layout(xaxis_title="Monthly Charges", yaxis_title="Customers")
fig.show()

In [ ]:
df["MonthlyChargeQuartile"] = pd.qcut(
    df["MonthlyCharges"],
    q=4,
    labels=["Q1 lowest", "Q2", "Q3", "Q4 highest"],
    duplicates="drop",
)

monthly_quartile_analysis = (
    df.groupby("MonthlyChargeQuartile", observed=True)
    .agg(Customers=("customerID", "count"), AvgMonthlyCharges=("MonthlyCharges", "mean"), ChurnRate=("ChurnFlag", "mean"))
    .reset_index()
)
monthly_quartile_analysis["AvgMonthlyCharges"] = monthly_quartile_analysis["AvgMonthlyCharges"].round(2)
monthly_quartile_analysis["ChurnRatePct"] = (monthly_quartile_analysis["ChurnRate"] * 100).round(2)
monthly_quartile_analysis

In [ ]:
fig = px.bar(
    monthly_quartile_analysis,
    x="MonthlyChargeQuartile",
    y="ChurnRatePct",
    color="MonthlyChargeQuartile",
    text="ChurnRatePct",
    title="Churn Rate by Monthly Charge Quartile",
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(showlegend=False, xaxis_title="Monthly Charge Quartile", yaxis_title="Churn Rate (%)")
fig.show()

In [ ]:
mean_charge_churned = df.loc[df["Churn"] == "Yes", "MonthlyCharges"].mean()
mean_charge_retained = df.loc[df["Churn"] == "No", "MonthlyCharges"].mean()
highest_quartile = monthly_quartile_analysis.sort_values("ChurnRate", ascending=False).iloc[0]

display(Markdown(
    f"""
**Observations**

- Average monthly charge for churned customers is **${mean_charge_churned:,.2f}**, compared with **${mean_charge_retained:,.2f}** for retained customers.
- The highest churn charge segment is **{highest_quartile['MonthlyChargeQuartile']}**, with a churn rate of **{highest_quartile['ChurnRate']:.1%}**.
- Pricing, perceived value, and bundled service experience should be explored together because high charges often interact with internet service and contract type.
"""
))

## 10. Correlation Analysis

Compute correlations across numeric fields and encoded categorical variables. This is exploratory only; encoded categorical correlations should be interpreted as directional signals, not causal relationships.

In [ ]:
correlation_base = df.drop(columns=["customerID"], errors="ignore").copy()

# One-hot encode categoricals for an exploratory correlation view.
encoded = pd.get_dummies(
    correlation_base.drop(columns=["Churn"], errors="ignore"),
    drop_first=True,
    dtype=int,
)

corr = encoded.corr(numeric_only=True)
churn_correlations = (
    corr["ChurnFlag"]
    .drop("ChurnFlag")
    .sort_values(key=lambda series: series.abs(), ascending=False)
    .reset_index()
)
churn_correlations.columns = ["Feature", "CorrelationWithChurn"]
churn_correlations.head(15)

In [ ]:
top_corr = churn_correlations.head(15).sort_values("CorrelationWithChurn")
fig = px.bar(
    top_corr,
    x="CorrelationWithChurn",
    y="Feature",
    orientation="h",
    color="CorrelationWithChurn",
    color_continuous_scale="RdBu",
    title="Top Absolute Correlations with Churn",
)
fig.update_layout(xaxis_title="Correlation with Churn", yaxis_title="Feature")
fig.show()

In [ ]:
selected_corr_columns = ["ChurnFlag", "tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
selected_corr = df[selected_corr_columns].corr(numeric_only=True)

fig = px.imshow(
    selected_corr,
    text_auto=".2f",
    color_continuous_scale="RdBu",
    zmin=-1,
    zmax=1,
    title="Numeric Feature Correlation Matrix",
)
fig.show()

In [ ]:
strongest_positive = churn_correlations.sort_values("CorrelationWithChurn", ascending=False).iloc[0]
strongest_negative = churn_correlations.sort_values("CorrelationWithChurn").iloc[0]
strongest_absolute = churn_correlations.iloc[0]

display(Markdown(
    f"""
**Observations**

- The strongest absolute churn signal in the encoded feature set is **{strongest_absolute['Feature']}** with correlation **{strongest_absolute['CorrelationWithChurn']:.2f}**.
- The strongest positive churn correlation is **{strongest_positive['Feature']}** with correlation **{strongest_positive['CorrelationWithChurn']:.2f}**.
- The strongest negative churn correlation is **{strongest_negative['Feature']}** with correlation **{strongest_negative['CorrelationWithChurn']:.2f}**.
- Tenure and total charges tend to move together because total charges accumulate over time.
- Correlation analysis supports the earlier visual pattern: short tenure and flexible contracts are important churn indicators.
"""
))

## 11. Interactive Segment Visualizations

Use Plotly interactions to compare churn by important customer and service segments. Hover, zoom, pan, and legend toggling are available in each chart.

In [ ]:
segment_columns = [
    "InternetService",
    "PaymentMethod",
    "PaperlessBilling",
    "SeniorCitizenLabel",
    "Partner",
    "Dependents",
    "TechSupport",
    "OnlineSecurity",
]

segment_frames = []
for column in segment_columns:
    summary = (
        df.groupby(column, observed=True)
        .agg(Customers=("customerID", "count"), ChurnRate=("ChurnFlag", "mean"))
        .reset_index()
        .rename(columns={column: "Segment"})
    )
    summary["Feature"] = column
    summary["ChurnRatePct"] = summary["ChurnRate"] * 100
    segment_frames.append(summary)

segment_analysis = pd.concat(segment_frames, ignore_index=True)
segment_analysis.head()

In [ ]:
fig = px.scatter(
    segment_analysis,
    x="Customers",
    y="ChurnRatePct",
    color="Feature",
    size="Customers",
    hover_name="Segment",
    facet_col="Feature",
    facet_col_wrap=2,
    title="Interactive Segment View: Size and Churn Rate",
    labels={"ChurnRatePct": "Churn Rate (%)"},
    height=900,
)
fig.update_yaxes(matches=None)
fig.update_xaxes(matches=None)
fig.show()

In [ ]:
fig = px.scatter(
    df,
    x="tenure",
    y="MonthlyCharges",
    color="Churn",
    hover_data=["Contract", "InternetService", "PaymentMethod", "TotalCharges"],
    opacity=0.65,
    title="Interactive Customer Map: Tenure vs Monthly Charges",
)
fig.update_layout(xaxis_title="Tenure (months)", yaxis_title="Monthly Charges")
fig.show()

In [ ]:
fig = px.box(
    df,
    x="Contract",
    y="MonthlyCharges",
    color="Churn",
    points="outliers",
    title="Monthly Charges by Contract Type and Churn Status",
)
fig.update_layout(xaxis_title="Contract Type", yaxis_title="Monthly Charges")
fig.show()

In [ ]:
high_risk_segments = (
    segment_analysis[segment_analysis["Customers"] >= 50]
    .sort_values("ChurnRate", ascending=False)
    .head(5)
)

display(Markdown(
    "**Observations**\n\n" +
    "\n".join(
        f"- **{row.Feature}: {row.Segment}** has a churn rate of **{row.ChurnRate:.1%}** across **{int(row.Customers):,} customers**."
        for row in high_risk_segments.itertuples(index=False)
    ) +
    "\n- These segment views help identify where retention campaigns should be prioritized before formal modeling."
))

## 12. Final EDA Summary

Key takeaways from this exploratory analysis.

In [ ]:
display(Markdown(
    f"""
**Final Observations**

- The dataset contains **{df.shape[0]:,} unique customer records** with no duplicate customer identifiers.
- The overall churn rate is **{churn_rate:.1%}**, making churn a meaningful but minority class.
- `TotalCharges` requires cleaning because blank strings in the raw CSV become numeric missing values.
- Month-to-month contracts show materially higher churn than longer-term contracts.
- Churn is concentrated among customers with shorter tenure, especially during the first year.
- Churned customers have higher average monthly charges than retained customers.
- Correlation and segment analysis point to contract type, tenure, internet service context, payment method, and support/security add-ons as important modeling candidates.

**Recommended next steps**

- Build a preprocessing pipeline for numeric imputation, categorical encoding, and target mapping.
- Use stratified train/test splitting because churn is imbalanced.
- Train baseline logistic regression and tree-based models, then compare ROC-AUC, recall, precision, and lift.
- Convert model outputs into customer-level churn risk scores for the Streamlit dashboard.
"""
))